# Comparing Five Machine Learning Models for Credit Risk Using Home Credit Data

This project uses `application_train.csv` from Kaggle's **Home Credit Default Risk** competition to predict whether an applicant will experience repayment difficulties (`TARGET=1`). It compares Logistic Regression, Decision Tree, Random Forest, XGBoost, and LightGBM.

The notebook covers data audit, cleaning, business feature engineering, training-only feature selection, resource-constrained parameter screening, final refitting, threshold analysis, risk deciles, probability calibration, statistical uncertainty, customer-segment diagnostics, random-split PSI, and illustrative cost sensitivity.

## Research Questions

- How much discrimination does each model provide by AUC, AR, KS, and PR-AUC?
- Are differences between the leading models distinguishable under paired bootstrap and DeLong inference?
- Are probability levels reliable after class weighting changes the effective training prior?
- Do customer segments show material performance dispersion?
- How sensitive is a threshold decision to illustrative false-approval and false-rejection cost ratios?

This is a model-development benchmark on one historical public dataset. It is not an out-of-time validation, a reject-inference study, or a production PD approval.


## 1. Environment and Dependency Check

The environment cell reports every directly used library version. Warnings are captured around model fitting and written to `outputs/warnings_log.csv`; they are not globally suppressed.


In [ ]:
import gc
import json
import os
import platform
import shutil
import sys
import textwrap
import time
import warnings
from pathlib import Path

import joblib
import lightgbm as lgb
import matplotlib
import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd
import scipy
import sklearn
import xgboost as xgb
from IPython.display import Markdown, display
from lightgbm import LGBMClassifier
from scipy import sparse
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GridSearchCV,
    ParameterSampler,
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

from credit_risk_validation import (
    bootstrap_metric_intervals,
    build_psi_table,
    calibration_diagnostics,
    cost_sensitivity_table,
    delong_auc_test,
    population_stability_index,
    prior_probability_correction,
    segment_performance_table,
)

warnings.simplefilter("default")

versions = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "matplotlib": matplotlib.__version__,
    "scikit-learn": sklearn.__version__,
    "xgboost": xgb.__version__,
    "lightgbm": lgb.__version__,
    "joblib": joblib.__version__,
    "nbformat": nbformat.__version__,
}
display(pd.Series(versions, name="version").to_frame())


## 2. Central Configuration

The default is a full experiment. For a quick check, set `HOME_CREDIT_DEBUG=1` before launching the notebook. Debug mode reads only the first 40,000 rows and reduces the search space. Full mode uses the entire application table, performs cross-validation tuning on a stratified training subset, and then fits each final model on the complete training set.


In [ ]:
PROJECT_ROOT = Path(os.getenv("HOME_CREDIT_PROJECT_ROOT", Path.cwd())).expanduser().resolve()
expected_notebook = PROJECT_ROOT / "home_credit_model_comparison.ipynb"
if not expected_notebook.exists():
    raise RuntimeError(
        "Run the notebook from the repository root or set HOME_CREDIT_PROJECT_ROOT explicitly. "
        f"Expected to find: {expected_notebook}"
    )

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "models"

RANDOM_STATE = 42
TARGET_COL = "TARGET"
ID_COL = "SK_ID_CURR"
TRAIN_SIZE = 0.60
MODEL_SELECTION_SIZE = 0.10
CALIBRATION_SIZE = 0.10
THRESHOLD_SIZE = 0.10
TEST_SIZE = 0.10
N_JOBS = -1
SEARCH_N_JOBS = 1

DEBUG_MODE = os.getenv("HOME_CREDIT_DEBUG", "0") == "1"
CLEAN_OUTPUTS = True
DEBUG_READ_ROWS = 40_000
TUNING_SAMPLE_SIZE = 12_000 if DEBUG_MODE else 60_000
CV_FOLDS = 2 if DEBUG_MODE else 5
BOOTSTRAP_REPLICATES = 100 if DEBUG_MODE else 500
HIGH_MISSING_THRESHOLD = 0.95
HIGH_CORRELATION_THRESHOLD = 0.98

SEARCH_ITERATIONS = {
    "decision_tree": 3 if DEBUG_MODE else 8,
    "random_forest": 2 if DEBUG_MODE else 8,
    "xgboost": 2 if DEBUG_MODE else 8,
    "lightgbm": 2 if DEBUG_MODE else 8,
}

for directory in (OUTPUT_DIR, FIGURE_DIR, MODEL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

if CLEAN_OUTPUTS:
    for artifact in FIGURE_DIR.glob("*.png"):
        artifact.unlink()
    for artifact in MODEL_DIR.glob("*.joblib"):
        artifact.unlink()
    for pattern in ("*.csv", "*.json", "*.md"):
        for artifact in OUTPUT_DIR.glob(pattern):
            artifact.unlink()

plt.rcParams.update({
    "figure.figsize": (9, 5.5),
    "axes.grid": True,
    "grid.alpha": 0.22,
    "font.sans-serif": ["Arial Unicode MS", "PingFang SC", "Heiti SC", "DejaVu Sans"],
    "axes.unicode_minus": False,
})

print({
    "DEBUG_MODE": DEBUG_MODE,
    "PROJECT_ROOT": str(PROJECT_ROOT),
    "TUNING_SAMPLE_SIZE": TUNING_SAMPLE_SIZE,
    "CV_FOLDS": CV_FOLDS,
    "BOOTSTRAP_REPLICATES": BOOTSTRAP_REPLICATES,
})


## 3. Shared Utilities

Shared utilities are defined near the beginning of the notebook and cover safe division, figure saving, stratified sampling, memory conversion, and JSON serialization.


In [ ]:
def safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    '''Safely divide and convert zero denominators or non-finite results to missing values.'''
    denominator = denominator.replace(0, np.nan)
    result = numerator / denominator
    return result.replace([np.inf, -np.inf], np.nan)


def save_figure(file_name: str, show: bool = False) -> Path:
    '''Apply a tight layout and save the current Matplotlib figure.'''
    path = FIGURE_DIR / file_name
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close()
    return path


def stratified_indices(y: pd.Series, max_rows: int, random_state: int = RANDOM_STATE) -> np.ndarray:
    '''Return stratified positional indices with at most max_rows observations.'''
    positions = np.arange(len(y))
    if len(y) <= max_rows:
        return positions
    selected, _ = train_test_split(
        positions,
        train_size=max_rows,
        stratify=np.asarray(y),
        random_state=random_state,
    )
    return np.sort(selected)


def as_float32(matrix):
    '''Convert preprocessed data to float32 while preserving sparse structure.'''
    if sparse.issparse(matrix):
        return matrix.astype(np.float32, copy=False).tocsr()
    return np.asarray(matrix, dtype=np.float32)


def json_safe(value):
    '''Convert NumPy scalars and related objects to JSON-serializable types.'''
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.generic):
        return value.item()
    return value


def slugify_model_name(name: str) -> str:
    '''Generate a stable model filename.'''
    return name.lower().replace(" ", "_").replace("-", "_")

## 4. Data Loading and Initial Audit

This repository deliberately models the application table only. Historical bureau, prior-application, and repayment tables are outside the implemented scope and are not presented as completed work. Kaggle's `application_test.csv` has no public labels, so mutually exclusive development and final-test partitions are drawn from `application_train.csv`.


In [ ]:
data_path = DATA_DIR / "application_train.csv"
if not data_path.exists():
    raise FileNotFoundError(
        f"Could not find {data_path}. Place application_train.csv in the project data/ directory."
    )

read_kwargs = {"low_memory": False}
if DEBUG_MODE:
    read_kwargs["nrows"] = DEBUG_READ_ROWS

load_started = time.perf_counter()
df_raw = pd.read_csv(data_path, **read_kwargs)
load_seconds = time.perf_counter() - load_started

required_columns = {TARGET_COL, ID_COL}
missing_required = required_columns - set(df_raw.columns)
if missing_required:
    raise ValueError(f"Missing required columns: {sorted(missing_required)}")

print(f"Data shape: {df_raw.shape}")
print(f"Load time: {load_seconds:.2f} seconds")
print(f"Memory usage: {df_raw.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
display(df_raw.head())

In [ ]:
target_counts = df_raw[TARGET_COL].value_counts(dropna=False).sort_index()
target_rates = df_raw[TARGET_COL].value_counts(normalize=True, dropna=False).sort_index()
numeric_columns_raw = df_raw.select_dtypes(include=np.number).columns.tolist()
categorical_columns_raw = df_raw.select_dtypes(exclude=np.number).columns.tolist()

basic_checks = {
    "target_missing": int(df_raw[TARGET_COL].isna().sum()),
    "duplicate_customer_ids": int(df_raw[ID_COL].duplicated().sum()),
    "duplicate_rows": int(df_raw.duplicated().sum()),
    "numeric_columns": len(numeric_columns_raw),
    "categorical_columns": len(categorical_columns_raw),
    "all_null_columns": int(df_raw.isna().all().sum()),
    "constant_columns_including_na": int(sum(df_raw[c].nunique(dropna=False) <= 1 for c in df_raw.columns)),
}

numeric_values = df_raw[numeric_columns_raw]
basic_checks["positive_or_negative_infinity"] = int(
    np.isinf(numeric_values.to_numpy(dtype=np.float64, copy=False)).sum()
)

display(pd.Series(basic_checks, name="value").to_frame())
display(pd.DataFrame({"count": target_counts, "rate": target_rates}))

assert basic_checks["target_missing"] == 0, "TARGET contains missing values"
assert set(target_counts.index).issubset({0, 1}), "TARGET is not binary"

### 4.1 Class Balance, Missingness, and Outliers

In addition to the target distribution, this section reports the 30 columns with the highest missing rates and applies an IQR diagnostic to numeric outliers. The outlier check is diagnostic only: no imputation or scaling statistics are fitted on the full dataset.


In [ ]:
fig, ax = plt.subplots()
ax.bar(target_counts.index.astype(str), target_counts.values, color=["#4C78A8", "#E45756"])
ax.set_title("TARGET Class Counts")
ax.set_xlabel("TARGET")
ax.set_ylabel("Number of Applicants")
for i, value in enumerate(target_counts.values):
    ax.text(i, value, f"{value:,}", ha="center", va="bottom")
save_figure("eda_target_count.png", show=True)

fig, ax = plt.subplots()
ax.bar(target_rates.index.astype(str), target_rates.values, color=["#72B7B2", "#F58518"])
ax.set_title("TARGET Class Proportions")
ax.set_xlabel("TARGET")
ax.set_ylabel("Applicant Proportion")
ax.set_ylim(0, 1)
for i, value in enumerate(target_rates.values):
    ax.text(i, value, f"{value:.2%}", ha="center", va="bottom")
save_figure("eda_target_rate.png", show=True)

missing_rates = df_raw.isna().mean().sort_values(ascending=False)
top_missing = missing_rates.head(30).sort_values()
fig, ax = plt.subplots(figsize=(10, 9))
ax.barh(top_missing.index, top_missing.values, color="#4C78A8")
ax.set_title("Top 30 Features by Missing Rate")
ax.set_xlabel("Missing Rate")
ax.set_ylabel("Feature")
save_figure("eda_top30_missing_rate.png", show=True)

numeric_for_outliers = [
    c for c in numeric_columns_raw
    if c not in {TARGET_COL, ID_COL} and df_raw[c].notna().sum() > 0
]
outlier_rows = []
for column in numeric_for_outliers:
    series = df_raw[column].replace([np.inf, -np.inf], np.nan).dropna()
    if series.empty:
        continue
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 3 * iqr, q3 + 3 * iqr
    outlier_rate = ((series < lower) | (series > upper)).mean() if iqr > 0 else 0.0
    outlier_rows.append({
        "feature": column,
        "min": series.min(),
        "p01": series.quantile(0.01),
        "p99": series.quantile(0.99),
        "max": series.max(),
        "extreme_iqr_rate": outlier_rate,
    })

outlier_summary = (
    pd.DataFrame(outlier_rows)
    .sort_values("extreme_iqr_rate", ascending=False)
    .reset_index(drop=True)
)
display(outlier_summary.head(20))

## 5. Data Cleaning

- Replace the Home Credit sentinel `DAYS_EMPLOYED=365243` with missing values.
- Replace positive and negative infinity with missing values.
- Remove entirely missing and single-value columns.
- Retain the applicant identifier only for traceability and exclude it before modeling.
- Fit medians, modes, and encoding mappings later inside training-only pipelines.


In [ ]:
df_clean = df_raw.copy()
cleaning_log = []

if "DAYS_EMPLOYED" in df_clean.columns:
    anomaly_count = int((df_clean["DAYS_EMPLOYED"] == 365243).sum())
    df_clean["DAYS_EMPLOYED"] = df_clean["DAYS_EMPLOYED"].replace(365243, np.nan)
    cleaning_log.append({"action": "replace_days_employed_365243", "affected": anomaly_count})

numeric_cols = df_clean.select_dtypes(include=np.number).columns
infinity_count = int(np.isinf(df_clean[numeric_cols].to_numpy(dtype=np.float64, copy=False)).sum())
df_clean[numeric_cols] = df_clean[numeric_cols].replace([np.inf, -np.inf], np.nan)
cleaning_log.append({"action": "replace_infinity", "affected": infinity_count})

all_null_columns = [c for c in df_clean.columns if df_clean[c].isna().all()]
constant_columns = [
    c for c in df_clean.columns
    if c not in {TARGET_COL, ID_COL} and df_clean[c].nunique(dropna=False) <= 1
]
pre_split_drop_columns = sorted(set(all_null_columns + constant_columns))
df_clean = df_clean.drop(columns=pre_split_drop_columns)
cleaning_log.append({"action": "drop_all_null_or_constant", "affected": len(pre_split_drop_columns)})

display(pd.DataFrame(cleaning_log))
display(pd.DataFrame({"dropped_column": pre_split_drop_columns}))

assert TARGET_COL in df_clean.columns
assert ID_COL in df_clean.columns
assert not np.isinf(df_clean.select_dtypes(include=np.number).to_numpy()).any()

## 6. Business Feature Engineering

Only a small set of interpretable ratios is created: credit burden, annuity burden, approximate term, goods-price coverage, employment stability, income per household member, and payment rate. Every division handles zero denominators and non-finite results.


In [ ]:
feature_formulas = {
    "CREDIT_INCOME_RATIO": ("AMT_CREDIT", "AMT_INCOME_TOTAL", "Credit relative to income; overall debt burden"),
    "ANNUITY_INCOME_RATIO": ("AMT_ANNUITY", "AMT_INCOME_TOTAL", "Annuity relative to income; repayment pressure"),
    "CREDIT_ANNUITY_RATIO": ("AMT_CREDIT", "AMT_ANNUITY", "Credit-to-annuity ratio; approximate repayment term"),
    "GOODS_CREDIT_RATIO": ("AMT_GOODS_PRICE", "AMT_CREDIT", "Goods price relative to credit amount"),
    "EMPLOYED_BIRTH_RATIO": ("DAYS_EMPLOYED", "DAYS_BIRTH", "Employment duration relative to age; employment stability"),
    "INCOME_PER_PERSON": ("AMT_INCOME_TOTAL", "CNT_FAM_MEMBERS", "Income per household member"),
    "PAYMENT_RATE": ("AMT_ANNUITY", "AMT_CREDIT", "Annuity-to-credit ratio; periodic payment burden"),
}

engineered_data = {}
engineered_features = []
feature_explanations = []
for new_feature, (numerator, denominator, explanation) in feature_formulas.items():
    if numerator in df_clean.columns and denominator in df_clean.columns:
        engineered_data[new_feature] = safe_divide(df_clean[numerator], df_clean[denominator])
        engineered_features.append(new_feature)
        feature_explanations.append({"feature": new_feature, "business_meaning": explanation})

engineered_frame = pd.DataFrame(engineered_data, index=df_clean.index).replace([np.inf, -np.inf], np.nan)
df_clean = pd.concat([df_clean, engineered_frame], axis=1).copy()
display(pd.DataFrame(feature_explanations))
print(f"Engineered business features: {len(engineered_features)}")

## 7. Independent Development and Evaluation Samples

The labeled sample is split into five mutually exclusive partitions:

- **Training (60%)**: preprocessing, feature selection, cross-validation, candidate fitting, and final model refitting.
- **Model selection (10%)**: boosted-model early stopping and candidate comparison only.
- **Calibration (10%)**: probability mapping only; internally divided for calibration-method comparison.
- **Threshold (10%)**: operating cutoff selection after the probability mapping is fixed.
- **Final test (10%)**: one final comparison and statistical inference.

The public data has no reliable application timestamp, so this design improves functional separation but does not replace out-of-time validation.


In [ ]:
X = df_clean.drop(columns=[TARGET_COL, ID_COL])
y = df_clean[TARGET_COL].astype(np.int8)
customer_ids = df_clean[ID_COL].copy()

X_train, X_holdout, y_train, y_holdout, id_train, id_holdout = train_test_split(
    X,
    y,
    customer_ids,
    test_size=1 - TRAIN_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

def split_one_quarter(X_source, y_source, id_source, seed):
    return train_test_split(
        X_source,
        y_source,
        id_source,
        test_size=0.25,
        stratify=y_source,
        random_state=seed,
    )

X_remaining, X_test, y_remaining, y_test, id_remaining, id_test = split_one_quarter(
    X_holdout, y_holdout, id_holdout, RANDOM_STATE + 1
)
X_remaining, X_threshold, y_remaining, y_threshold, id_remaining, id_threshold = train_test_split(
    X_remaining,
    y_remaining,
    id_remaining,
    test_size=1 / 3,
    stratify=y_remaining,
    random_state=RANDOM_STATE + 2,
)
X_selection, X_calibration, y_selection, y_calibration, id_selection, id_calibration = train_test_split(
    X_remaining,
    y_remaining,
    id_remaining,
    test_size=0.5,
    stratify=y_remaining,
    random_state=RANDOM_STATE + 3,
)

split_objects = {
    "train": (X_train, y_train, id_train),
    "model_selection": (X_selection, y_selection, id_selection),
    "calibration": (X_calibration, y_calibration, id_calibration),
    "threshold": (X_threshold, y_threshold, id_threshold),
    "test": (X_test, y_test, id_test),
}
split_summary = pd.DataFrame([
    {
        "split": name,
        "rows": len(values[1]),
        "bad_count": int(values[1].sum()),
        "bad_rate": float(values[1].mean()),
    }
    for name, values in split_objects.items()
])
display(split_summary)

id_sets = {name: set(values[2]) for name, values in split_objects.items()}
for left_index, left_name in enumerate(id_sets):
    for right_name in list(id_sets)[left_index + 1:]:
        assert id_sets[left_name].isdisjoint(id_sets[right_name])
assert np.isclose(sum(len(values[1]) for values in split_objects.values()), len(y))
assert max(split_summary["bad_rate"]) - min(split_summary["bad_rate"]) < 0.003

del X_holdout, y_holdout, id_holdout, X_remaining, y_remaining, id_remaining
gc.collect()


## 8. Training-Only Feature Selection

Only the training set is used to decide which columns to remove:

1. More than 95% missing in the training set.
2. No variation in the training set.
3. For numeric pairs with absolute correlation above 0.98, keep the earlier column and remove the later one.

This demonstrates feature-selection logic without using validation or test statistics.


In [ ]:
train_missing_rates = X_train.isna().mean()
high_missing_columns = train_missing_rates[train_missing_rates > HIGH_MISSING_THRESHOLD].index.tolist()
train_constant_columns = [c for c in X_train.columns if X_train[c].nunique(dropna=False) <= 1]

numeric_train_columns = X_train.select_dtypes(include=np.number).columns.tolist()
correlation_matrix = X_train[numeric_train_columns].corr().abs()
upper_triangle = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)
high_correlation_columns = [
    column for column in upper_triangle.columns
    if (upper_triangle[column] > HIGH_CORRELATION_THRESHOLD).any()
]

selection_drop_columns = sorted(set(
    high_missing_columns + train_constant_columns + high_correlation_columns
))
selected_columns = [c for c in X_train.columns if c not in selection_drop_columns]

X_train = X_train[selected_columns].copy()
X_selection = X_selection[selected_columns].copy()
X_calibration = X_calibration[selected_columns].copy()
X_threshold = X_threshold[selected_columns].copy()
X_test = X_test[selected_columns].copy()

selection_summary = pd.DataFrame([
    {"rule": "missing_rate_gt_threshold", "threshold": HIGH_MISSING_THRESHOLD, "dropped": len(high_missing_columns)},
    {"rule": "constant_in_training", "threshold": np.nan, "dropped": len(train_constant_columns)},
    {"rule": "absolute_correlation_gt_threshold", "threshold": HIGH_CORRELATION_THRESHOLD, "dropped": len(high_correlation_columns)},
    {"rule": "total_unique_dropped", "threshold": np.nan, "dropped": len(selection_drop_columns)},
])
display(selection_summary)
display(pd.DataFrame({"dropped_feature": selection_drop_columns}).head(50))
print(f"Features before selection: {X.shape[1]}; features after selection: {len(selected_columns)}")

pd.DataFrame({"selected_feature": selected_columns}).to_csv(
    OUTPUT_DIR / "selected_features.csv", index=False
)
pd.DataFrame({"dropped_feature": selection_drop_columns}).to_csv(
    OUTPUT_DIR / "dropped_features.csv", index=False
)


## 9. Training-Fitted Preprocessing Pipelines

Separate preprocessors are built for Logistic Regression and tree models. Numeric imputation, scaling, categorical imputation, and One-Hot category levels are fitted only on the training partition. Model-selection, calibration, threshold, and final-test partitions call `transform` only.


In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_lr_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False)),
])
numeric_tree_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True, dtype=np.float32)),
])

logistic_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_lr_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
    sparse_threshold=1.0,
    verbose_feature_names_out=False,
)
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_tree_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
    sparse_threshold=1.0,
    verbose_feature_names_out=False,
)

transform_started = time.perf_counter()
X_train_logistic = as_float32(logistic_preprocessor.fit_transform(X_train))
X_selection_logistic = as_float32(logistic_preprocessor.transform(X_selection))
X_calibration_logistic = as_float32(logistic_preprocessor.transform(X_calibration))
X_threshold_logistic = as_float32(logistic_preprocessor.transform(X_threshold))
X_test_logistic = as_float32(logistic_preprocessor.transform(X_test))

X_train_tree = as_float32(tree_preprocessor.fit_transform(X_train))
X_selection_tree = as_float32(tree_preprocessor.transform(X_selection))
X_calibration_tree = as_float32(tree_preprocessor.transform(X_calibration))
X_threshold_tree = as_float32(tree_preprocessor.transform(X_threshold))
X_test_tree = as_float32(tree_preprocessor.transform(X_test))
transform_seconds = time.perf_counter() - transform_started

logistic_feature_names = np.asarray(logistic_preprocessor.get_feature_names_out())
tree_feature_names = np.asarray(tree_preprocessor.get_feature_names_out())
joblib.dump(logistic_preprocessor, MODEL_DIR / "logistic_preprocessor.joblib")
joblib.dump(tree_preprocessor, MODEL_DIR / "tree_preprocessor.joblib")

encoded_summary = pd.DataFrame([
    {"pipeline": "logistic", "train_shape": str(X_train_logistic.shape), "sparse": sparse.issparse(X_train_logistic)},
    {"pipeline": "tree", "train_shape": str(X_train_tree.shape), "sparse": sparse.issparse(X_train_tree)},
])
display(encoded_summary)
print(f"Total preprocessing time: {transform_seconds:.2f} seconds")

assert TARGET_COL not in logistic_feature_names
assert ID_COL not in logistic_feature_names
assert len({
    matrix.shape[1] for matrix in (
        X_train_logistic, X_selection_logistic, X_calibration_logistic,
        X_threshold_logistic, X_test_logistic,
    )
}) == 1
assert len({
    matrix.shape[1] for matrix in (
        X_train_tree, X_selection_tree, X_calibration_tree, X_threshold_tree, X_test_tree,
    )
}) == 1
assert X_train_tree.shape[1] < 5_000, "Unexpected One-Hot feature explosion"


## 10. Unified Evaluation, Threshold, and Tuning Functions

AUC, AR, KS, and PR-AUC are calculated from continuous scores. Hyperparameters and boosting iterations use the model-selection sample; calibration mappings use only the calibration sample; operating thresholds use only the threshold sample. The final test sample is evaluated after those choices are fixed.


In [ ]:
def select_thresholds(y_true, y_prob, recall_targets=(0.70, 0.80)) -> dict[str, float]:
    '''Calculate maximum-F1, maximum-Youden, and target-recall thresholds from validation probabilities.'''
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    precision, recall, pr_thresholds = precision_recall_curve(y_true, y_prob)
    if len(pr_thresholds) == 0:
        raise ValueError("Cannot select a threshold from a single unique predicted probability")
    denominator = precision[:-1] + recall[:-1]
    f1_values = np.divide(
        2 * precision[:-1] * recall[:-1],
        denominator,
        out=np.zeros_like(denominator),
        where=denominator > 0,
    )
    max_f1_threshold = float(pr_thresholds[int(np.nanargmax(f1_values))])

    fpr, tpr, roc_thresholds = roc_curve(y_true, y_prob)
    finite_mask = np.isfinite(roc_thresholds)
    youden_values = tpr[finite_mask] - fpr[finite_mask]
    youden_threshold = float(roc_thresholds[finite_mask][int(np.argmax(youden_values))])

    thresholds = {
        "default_0.5": 0.5,
        "max_f1": max_f1_threshold,
        "max_youden": youden_threshold,
    }
    for target in recall_targets:
        valid = np.where(recall[:-1] >= target)[0]
        if len(valid):
            best_position = valid[int(np.argmax(precision[:-1][valid]))]
            thresholds[f"target_recall_{int(target * 100)}"] = float(pr_thresholds[best_position])
    return thresholds


def evaluate_model(model_name, y_true, y_prob, threshold=0.5) -> dict[str, float]:
    '''Calculate unified probability and classification metrics for binary credit risk.'''
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)
    if len(y_true) != len(y_prob):
        raise ValueError("y_true and y_prob have different lengths")
    if not np.isfinite(y_prob).all():
        raise ValueError(f"{model_name} contains NaN or infinite predicted probabilities")

    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    auc = roc_auc_score(y_true, y_prob)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    ks = float(np.max(tpr - fpr))
    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    metrics = {
        "Model": model_name,
        "Threshold": float(threshold),
        "AUC": float(auc),
        "AR": float(2 * auc - 1),
        "KS": ks,
        "PR-AUC": float(average_precision_score(y_true, y_prob)),
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "Recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "F1": float(f1_score(y_true, y_pred, zero_division=0)),
        "Specificity": float(specificity),
        "Brier": float(brier_score_loss(y_true, y_prob)),
        "LogLoss": float(log_loss(y_true, np.clip(y_prob, 1e-7, 1 - 1e-7))),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        "PassRate": float((y_pred == 0).mean()),
        "RejectRate": float((y_pred == 1).mean()),
        "BadCaptureRate": float(tp / (tp + fn)) if (tp + fn) else np.nan,
    }

    assert 0 <= metrics["AUC"] <= 1
    assert -1 <= metrics["AR"] <= 1
    assert 0 <= metrics["KS"] <= 1
    assert 0 <= metrics["PR-AUC"] <= 1
    return metrics


def threshold_operating_table(model_name, y_true, y_prob, thresholds) -> pd.DataFrame:
    '''Return operating metrics for multiple thresholds of the same model.'''
    rows = []
    for threshold_name, threshold in thresholds.items():
        metrics = evaluate_model(model_name, y_true, y_prob, threshold)
        rows.append({
            "Model": model_name,
            "ThresholdName": threshold_name,
            "Threshold": threshold,
            "Precision": metrics["Precision"],
            "Recall": metrics["Recall"],
            "F1": metrics["F1"],
            "Specificity": metrics["Specificity"],
            "PassRate": metrics["PassRate"],
            "RejectRate": metrics["RejectRate"],
            "BadCaptureRate": metrics["BadCaptureRate"],
        })
    return pd.DataFrame(rows)


def positive_probability(model, matrix) -> np.ndarray:
    '''Return predicted probabilities for the positive class.'''
    if isinstance(model, LGBMClassifier) and hasattr(model, "booster_"):
        probability = model.booster_.predict(matrix)
    else:
        probability = model.predict_proba(matrix)[:, 1]
    return np.asarray(probability, dtype=float)

In [ ]:
cv_strategy = StratifiedKFold(
    n_splits=CV_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

tuning_positions = stratified_indices(y_train, TUNING_SAMPLE_SIZE)
X_tune_logistic = X_train_logistic[tuning_positions]
X_tune_tree = X_train_tree[tuning_positions]
y_tune = y_train.iloc[tuning_positions].to_numpy()

print({
    "tuning_rows": len(y_tune),
    "tuning_bad_count": int(y_tune.sum()),
    "tuning_bad_rate": float(y_tune.mean()),
    "cv_folds": CV_FOLDS,
})

In [ ]:
warning_records = []


def run_with_warning_audit(context, callable_):
    '''Run one fitting operation and retain every Python warning for review.'''
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        result = callable_()
    for warning in caught:
        warning_records.append({
            "Context": context,
            "Category": warning.category.__name__,
            "Message": str(warning.message),
        })
    return result


def tune_sklearn_model(
    estimator,
    parameter_space,
    X_tune,
    y_tune,
    X_full,
    y_full,
    search_kind="random",
    n_iter=None,
):
    '''Screen parameters on a tuning subset, then time a clean fit on the full training set.'''
    if search_kind == "grid":
        search = GridSearchCV(
            estimator=estimator,
            param_grid=parameter_space,
            scoring="roc_auc",
            cv=cv_strategy,
            n_jobs=SEARCH_N_JOBS,
            refit=False,
            error_score="raise",
            return_train_score=True,
        )
    else:
        search = RandomizedSearchCV(
            estimator=estimator,
            param_distributions=parameter_space,
            n_iter=n_iter,
            scoring="roc_auc",
            cv=cv_strategy,
            n_jobs=SEARCH_N_JOBS,
            refit=False,
            random_state=RANDOM_STATE,
            error_score="raise",
            return_train_score=True,
        )

    tuning_started = time.perf_counter()
    run_with_warning_audit("parameter_search", lambda: search.fit(X_tune, y_tune))
    tuning_seconds = time.perf_counter() - tuning_started

    final_model = clone(estimator).set_params(**search.best_params_)
    training_started = time.perf_counter()
    run_with_warning_audit("final_full_training_fit", lambda: final_model.fit(X_full, np.asarray(y_full)))
    training_seconds = time.perf_counter() - training_started

    return final_model, {
        "best_params": json_safe(search.best_params_),
        "best_search_auc": float(search.best_score_),
        "tuning_seconds": tuning_seconds,
        "training_seconds": training_seconds,
        "candidate_count": len(search.cv_results_["params"]),
        "timing_definition": "final_clean_fit_on_complete_training_partition",
    }


def tune_xgboost(X_tune_matrix, y_tune_values, X_selection_matrix, y_selection_values, X_full, y_full):
    '''Screen XGBoost candidates with early stopping, then refit the selected iteration count.'''
    negative_count = int((np.asarray(y_tune_values) == 0).sum())
    positive_count = int((np.asarray(y_tune_values) == 1).sum())
    scale_pos_weight = negative_count / positive_count
    parameter_space = {
        "learning_rate": [0.02, 0.04, 0.06],
        "max_depth": [3, 4, 5],
        "min_child_weight": [5, 15, 30],
        "subsample": [0.75, 0.9, 1.0],
        "colsample_bytree": [0.65, 0.8, 1.0],
        "reg_alpha": [0.0, 0.1, 0.5],
        "reg_lambda": [1.0, 5.0, 10.0],
        "gamma": [0.0, 0.1, 0.3],
    }
    candidates = list(ParameterSampler(
        parameter_space,
        n_iter=SEARCH_ITERATIONS["xgboost"],
        random_state=RANDOM_STATE,
    ))
    maximum_estimators = 300 if DEBUG_MODE else 2_400
    early_stopping_rounds = 25 if DEBUG_MODE else 80
    tuning_started = time.perf_counter()
    candidate_rows = []
    best = None

    for candidate_number, params in enumerate(candidates, start=1):
        candidate_model = xgb.XGBClassifier(
            n_estimators=maximum_estimators,
            objective="binary:logistic",
            eval_metric="auc",
            tree_method="hist",
            max_bin=255,
            scale_pos_weight=scale_pos_weight,
            early_stopping_rounds=early_stopping_rounds,
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
            **params,
        )
        fit_started = time.perf_counter()
        run_with_warning_audit(
            f"xgboost_candidate_{candidate_number}",
            lambda model=candidate_model: model.fit(
                X_tune_matrix,
                np.asarray(y_tune_values),
                eval_set=[(X_selection_matrix, np.asarray(y_selection_values))],
                verbose=False,
            ),
        )
        fit_seconds = time.perf_counter() - fit_started
        selection_probability = positive_probability(candidate_model, X_selection_matrix)
        selection_auc = roc_auc_score(y_selection_values, selection_probability)
        best_iteration = int(candidate_model.best_iteration)
        row = {
            "candidate": candidate_number,
            "selection_auc": selection_auc,
            "best_iteration_zero_based": best_iteration,
            "final_estimators": best_iteration + 1,
            "reached_iteration_boundary": best_iteration >= maximum_estimators - early_stopping_rounds - 1,
            "fit_seconds": fit_seconds,
            "params": json.dumps(json_safe(params), ensure_ascii=False, sort_keys=True),
        }
        candidate_rows.append(row)
        if best is None or selection_auc > best["selection_auc"]:
            best = row | {"params_object": params}

    final_params = best["params_object"] | {
        "n_estimators": int(best["final_estimators"]),
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "tree_method": "hist",
        "max_bin": 255,
        "scale_pos_weight": scale_pos_weight,
        "random_state": RANDOM_STATE,
        "n_jobs": N_JOBS,
    }
    final_model = xgb.XGBClassifier(**final_params)
    training_started = time.perf_counter()
    run_with_warning_audit(
        "xgboost_final_full_training_fit",
        lambda: final_model.fit(X_full, np.asarray(y_full), verbose=False),
    )
    training_seconds = time.perf_counter() - training_started
    return final_model, {
        "best_params": json_safe(final_params),
        "best_search_auc": float(best["selection_auc"]),
        "tuning_seconds": time.perf_counter() - tuning_started - training_seconds,
        "training_seconds": training_seconds,
        "candidate_count": len(candidates),
        "candidate_results": pd.DataFrame(candidate_rows),
        "timing_definition": "final_clean_fit_on_complete_training_partition",
        "early_stopping_boundary_reached": bool(best["reached_iteration_boundary"]),
    }


def tune_lightgbm(X_tune_matrix, y_tune_values, X_selection_matrix, y_selection_values, X_full, y_full):
    '''Screen LightGBM candidates with active bagging, then refit the selected iteration count.'''
    negative_count = int((np.asarray(y_tune_values) == 0).sum())
    positive_count = int((np.asarray(y_tune_values) == 1).sum())
    scale_pos_weight = negative_count / positive_count
    parameter_space = {
        "learning_rate": [0.02, 0.04, 0.06],
        "num_leaves": [15, 31, 63],
        "max_depth": [-1, 5, 7],
        "min_child_samples": [30, 80, 150],
        "subsample": [0.75, 0.9, 1.0],
        "colsample_bytree": [0.65, 0.8, 1.0],
        "reg_alpha": [0.0, 0.1, 0.5],
        "reg_lambda": [0.0, 1.0, 5.0],
    }
    candidates = list(ParameterSampler(
        parameter_space,
        n_iter=SEARCH_ITERATIONS["lightgbm"],
        random_state=RANDOM_STATE,
    ))
    maximum_estimators = 400 if DEBUG_MODE else 2_500
    early_stopping_rounds = 25 if DEBUG_MODE else 80
    tuning_started = time.perf_counter()
    candidate_rows = []
    best = None

    for candidate_number, params in enumerate(candidates, start=1):
        effective_params = params | {"subsample_freq": 1}
        candidate_model = LGBMClassifier(
            n_estimators=maximum_estimators,
            objective="binary",
            metric="auc",
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
            verbosity=-1,
            **effective_params,
        )
        fit_started = time.perf_counter()
        run_with_warning_audit(
            f"lightgbm_candidate_{candidate_number}",
            lambda model=candidate_model: model.fit(
                X_tune_matrix,
                np.asarray(y_tune_values),
                eval_set=[(X_selection_matrix, np.asarray(y_selection_values))],
                eval_metric="auc",
                callbacks=[lgb.early_stopping(early_stopping_rounds, verbose=False)],
            ),
        )
        fit_seconds = time.perf_counter() - fit_started
        selection_probability = positive_probability(candidate_model, X_selection_matrix)
        selection_auc = roc_auc_score(y_selection_values, selection_probability)
        best_iteration = int(candidate_model.best_iteration_)
        row = {
            "candidate": candidate_number,
            "selection_auc": selection_auc,
            "best_iteration": best_iteration,
            "final_estimators": best_iteration,
            "reached_iteration_boundary": best_iteration >= maximum_estimators - early_stopping_rounds,
            "fit_seconds": fit_seconds,
            "params": json.dumps(json_safe(effective_params), ensure_ascii=False, sort_keys=True),
        }
        candidate_rows.append(row)
        if best is None or selection_auc > best["selection_auc"]:
            best = row | {"params_object": effective_params}

    final_params = best["params_object"] | {
        "n_estimators": int(best["final_estimators"]),
        "objective": "binary",
        "metric": "auc",
        "scale_pos_weight": scale_pos_weight,
        "random_state": RANDOM_STATE,
        "n_jobs": N_JOBS,
        "verbosity": -1,
    }
    final_model = LGBMClassifier(**final_params)
    training_started = time.perf_counter()
    run_with_warning_audit(
        "lightgbm_final_full_training_fit",
        lambda: final_model.fit(X_full, np.asarray(y_full)),
    )
    training_seconds = time.perf_counter() - training_started
    return final_model, {
        "best_params": json_safe(final_params),
        "best_search_auc": float(best["selection_auc"]),
        "tuning_seconds": time.perf_counter() - tuning_started - training_seconds,
        "training_seconds": training_seconds,
        "candidate_count": len(candidates),
        "candidate_results": pd.DataFrame(candidate_rows),
        "timing_definition": "final_clean_fit_on_complete_training_partition",
        "early_stopping_boundary_reached": bool(best["reached_iteration_boundary"]),
    }


## 11. Resource-Constrained Parameter Screening and Final Refitting

Logistic Regression uses a small grid; Decision Tree and Random Forest use limited randomized searches. XGBoost and LightGBM each screen eight randomized candidates in the full run. This is intentionally described as **limited parameter screening**, not exhaustive optimization. Every selected model is then fitted once on the complete training partition, and that clean final fit defines the reported training time.


### 11.1 Logistic Regression

Logistic Regression provides an interpretable credit-scoring baseline. The `saga` solver supports both L1 and L2 penalties and is better suited to large sparse One-Hot matrices while avoiding invalid solver/penalty combinations.


In [ ]:
models = {}
training_info = {}

logistic_base = LogisticRegression(
    class_weight="balanced",
    max_iter=3_000,
    solver="saga",
    l1_ratio=0,
    tol=5e-3,
    random_state=RANDOM_STATE,
)
logistic_grid = {
    "C": [0.1, 1.0] if DEBUG_MODE else [0.01, 0.1, 1.0],
    "l1_ratio": [0.0, 1.0],
    "class_weight": ["balanced"],
}
models["Logistic Regression"], training_info["Logistic Regression"] = tune_sklearn_model(
    logistic_base,
    logistic_grid,
    X_tune_logistic,
    y_tune,
    X_train_logistic,
    y_train,
    search_kind="grid",
)
print(training_info["Logistic Regression"])


### 11.2 Decision Tree

A single tree is used to examine nonlinear relationships and feature interactions. Depth, leaf size, and pruning parameters control overfitting.


In [ ]:
decision_tree_base = DecisionTreeClassifier(random_state=RANDOM_STATE)
decision_tree_space = {
    "criterion": ["gini", "entropy", "log_loss"],
    "max_depth": [4, 6, 8, 12, None],
    "min_samples_split": [10, 50, 200],
    "min_samples_leaf": [20, 100, 500],
    "max_features": [None, "sqrt", 0.7],
    "class_weight": [None, "balanced"],
    "ccp_alpha": [0.0, 1e-5, 1e-4],
}
models["Decision Tree"], training_info["Decision Tree"] = tune_sklearn_model(
    decision_tree_base,
    decision_tree_space,
    X_tune_tree,
    y_tune,
    X_train_tree,
    y_train,
    search_kind="random",
    n_iter=SEARCH_ITERATIONS["decision_tree"],
)
tree_model = models["Decision Tree"]
training_info["Decision Tree"]["tree_depth"] = int(tree_model.get_depth())
training_info["Decision Tree"]["leaf_count"] = int(tree_model.get_n_leaves())
print(training_info["Decision Tree"])

### 11.3 Random Forest

Random Forest uses bagging to reduce the variance of a single tree. Tree depth and leaf size are constrained to control model size, and the OOB score is retained as an additional diagnostic.


In [ ]:
random_forest_base = RandomForestClassifier(
    class_weight="balanced_subsample",
    bootstrap=True,
    oob_score=True,
    n_jobs=N_JOBS,
    random_state=RANDOM_STATE,
)
random_forest_space = {
    "n_estimators": [120, 180] if DEBUG_MODE else [180, 250, 320],
    "max_depth": [8, 12, 16],
    "min_samples_split": [10, 50, 150],
    "min_samples_leaf": [5, 20, 50],
    "max_features": ["sqrt", 0.5],
    "class_weight": ["balanced", "balanced_subsample"],
}
models["Random Forest"], training_info["Random Forest"] = tune_sklearn_model(
    random_forest_base,
    random_forest_space,
    X_tune_tree,
    y_tune,
    X_train_tree,
    y_train,
    search_kind="random",
    n_iter=SEARCH_ITERATIONS["random_forest"],
)
training_info["Random Forest"]["oob_score"] = float(models["Random Forest"].oob_score_)
print(training_info["Random Forest"])

### 11.4 XGBoost

Candidate models use the training subset and model-selection partition for early stopping. The selected iteration count is inspected for boundary contact, then a fresh model is fitted on the complete training partition without reusing the model-selection data.


In [ ]:
models["XGBoost"], training_info["XGBoost"] = tune_xgboost(
    X_tune_tree,
    y_tune,
    X_selection_tree,
    y_selection,
    X_train_tree,
    y_train,
)
display(training_info["XGBoost"]["candidate_results"])


### 11.5 LightGBM

Candidate models use leaf, depth, minimum-child, column-sampling, row-sampling, and regularization controls. `subsample_freq=1` explicitly activates row bagging whenever `subsample<1`. The selected iteration count is then used in a fresh full-training fit.


In [ ]:
models["LightGBM"], training_info["LightGBM"] = tune_lightgbm(
    X_tune_tree,
    y_tune,
    X_selection_tree,
    y_selection,
    X_train_tree,
    y_train,
)
display(training_info["LightGBM"]["candidate_results"])

assert set(models) == {
    "Logistic Regression", "Decision Tree", "Random Forest", "XGBoost", "LightGBM"
}


## 12. Model Diagnostics and Final Test Evaluation

Train/model-selection AUC gaps are reported as continuous diagnostics, not converted into an arbitrary pass/fail flag. Thresholds are selected on the independent threshold partition and then applied once to the final test partition.


In [ ]:
matrix_sets = {
    "Logistic Regression": (X_train_logistic, X_selection_logistic, X_calibration_logistic, X_threshold_logistic, X_test_logistic),
    "Decision Tree": (X_train_tree, X_selection_tree, X_calibration_tree, X_threshold_tree, X_test_tree),
    "Random Forest": (X_train_tree, X_selection_tree, X_calibration_tree, X_threshold_tree, X_test_tree),
    "XGBoost": (X_train_tree, X_selection_tree, X_calibration_tree, X_threshold_tree, X_test_tree),
    "LightGBM": (X_train_tree, X_selection_tree, X_calibration_tree, X_threshold_tree, X_test_tree),
}

probabilities = {}
threshold_tables = []
selected_metric_rows = []
default_metric_rows = []
gap_rows = []
best_parameter_rows = []

for model_name, model in models.items():
    train_matrix, selection_matrix, calibration_matrix, threshold_matrix, test_matrix = matrix_sets[model_name]
    split_matrices = {
        "train": train_matrix,
        "model_selection": selection_matrix,
        "calibration": calibration_matrix,
        "threshold": threshold_matrix,
        "test": test_matrix,
    }
    model_probabilities = {}
    for split_name, matrix in split_matrices.items():
        prediction_started = time.perf_counter()
        values = positive_probability(model, matrix)
        elapsed = time.perf_counter() - prediction_started
        if len(values) == 0 or not np.isfinite(values).all() or np.std(values) < 1e-8:
            raise ValueError(f"{model_name} produced invalid {split_name} probabilities")
        model_probabilities[split_name] = values
        if split_name == "test":
            prediction_seconds = elapsed
    probabilities[model_name] = model_probabilities

    thresholds = select_thresholds(y_threshold, model_probabilities["threshold"])
    threshold_tables.append(
        threshold_operating_table(model_name, y_threshold, model_probabilities["threshold"], thresholds)
        .assign(Dataset="threshold")
    )
    selected_threshold = thresholds["max_f1"]
    default_metrics = evaluate_model(model_name, y_test, model_probabilities["test"], threshold=0.5)
    selected_metrics = evaluate_model(model_name, y_test, model_probabilities["test"], threshold=selected_threshold)
    default_metrics["ThresholdName"] = "default_0.5"
    selected_metrics["ThresholdName"] = "threshold_max_f1"
    selected_metrics["TrainTime"] = training_info[model_name]["training_seconds"]
    selected_metrics["PredictTime"] = prediction_seconds
    selected_metrics["TuneTime"] = training_info[model_name]["tuning_seconds"]
    default_metric_rows.append(default_metrics)
    selected_metric_rows.append(selected_metrics)

    train_auc = roc_auc_score(y_train, model_probabilities["train"])
    selection_auc = roc_auc_score(y_selection, model_probabilities["model_selection"])
    gap_rows.append({
        "Model": model_name,
        "TrainAUC": train_auc,
        "ModelSelectionAUC": selection_auc,
        "TestAUC": selected_metrics["AUC"],
        "TrainSelectionGap": train_auc - selection_auc,
        "Interpretation": "Gap is descriptive; stability requires resampling and out-of-time evidence.",
    })

    best_parameter_rows.append({
        "Model": model_name,
        "BestParameters": json.dumps(json_safe(training_info[model_name]["best_params"]), ensure_ascii=False, sort_keys=True),
        "BestSearchAUC": training_info[model_name]["best_search_auc"],
        "CandidateCount": training_info[model_name]["candidate_count"],
        "TuneTime": training_info[model_name]["tuning_seconds"],
        "TrainTime": training_info[model_name]["training_seconds"],
        "TimingDefinition": training_info[model_name]["timing_definition"],
        "EarlyStoppingBoundaryReached": training_info[model_name].get("early_stopping_boundary_reached", np.nan),
    })

    joblib.dump(model, MODEL_DIR / f"{slugify_model_name(model_name)}.joblib")

threshold_analysis = pd.concat(threshold_tables, ignore_index=True)
model_metrics = pd.DataFrame(selected_metric_rows)
default_threshold_metrics = pd.DataFrame(default_metric_rows)
performance_gaps = pd.DataFrame(gap_rows)
best_parameters = pd.DataFrame(best_parameter_rows)

metric_columns = [
    "Model", "AUC", "AR", "KS", "PR-AUC", "Precision", "Recall", "F1",
    "Specificity", "Brier", "LogLoss", "TrainTime", "PredictTime", "TuneTime",
    "Threshold", "PassRate", "RejectRate", "BadCaptureRate",
]
model_metrics = model_metrics[metric_columns].sort_values("AUC", ascending=False).reset_index(drop=True)
display(model_metrics.round(4))
display(performance_gaps.round(4))
print(f"Random Forest OOB Score: {training_info['Random Forest'].get('oob_score', np.nan):.4f}")


### 12.1 Save Core Result Tables

In [ ]:
model_metrics.to_csv(OUTPUT_DIR / "model_metrics.csv", index=False)
default_threshold_metrics.to_csv(OUTPUT_DIR / "model_metrics_default_threshold.csv", index=False)
best_parameters.to_csv(OUTPUT_DIR / "best_parameters.csv", index=False)
threshold_analysis.to_csv(OUTPUT_DIR / "threshold_analysis.csv", index=False)
performance_gaps.to_csv(OUTPUT_DIR / "train_selection_performance_gap.csv", index=False)
split_summary.to_csv(OUTPUT_DIR / "data_split_summary.csv", index=False)
training_info["XGBoost"]["candidate_results"].to_csv(OUTPUT_DIR / "xgboost_candidate_results.csv", index=False)
training_info["LightGBM"]["candidate_results"].to_csv(OUTPUT_DIR / "lightgbm_candidate_results.csv", index=False)

test_predictions = pd.DataFrame({
    ID_COL: np.asarray(id_test),
    TARGET_COL: np.asarray(y_test),
    **{model: values["test"] for model, values in probabilities.items()},
})
test_predictions.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)

run_metadata = {
    "project_root": str(PROJECT_ROOT),
    "random_state": RANDOM_STATE,
    "debug_mode": DEBUG_MODE,
    "source_rows": int(len(df_raw)),
    "source_columns": int(df_raw.shape[1]),
    "selected_raw_features": int(len(selected_columns)),
    "encoded_features": int(X_train_tree.shape[1]),
    "cv_folds": CV_FOLDS,
    "tuning_sample_size": int(len(y_tune)),
    "bootstrap_replicates": BOOTSTRAP_REPLICATES,
    "split_purpose": {
        "train": "preprocessing, feature selection, tuning CV, and final fit",
        "model_selection": "boosting candidate selection and early stopping",
        "calibration": "probability mapping",
        "threshold": "operating cutoff selection",
        "test": "one final evaluation and inference",
    },
    "library_versions": versions,
}
(OUTPUT_DIR / "run_metadata.json").write_text(
    json.dumps(run_metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)


## 13. Per-Model Charts and Interpretability

Each model saves ROC, PR, KS, confusion matrix, calibration, probability distribution, and top-20 feature charts. Logistic Regression uses signed coefficients; the other models use built-in feature importance.


In [ ]:
def plot_single_roc(model_name, y_true, y_prob):
    '''Save a per-model ROC curve.'''
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    plt.figure()
    plt.plot(fpr, tpr, label=f"AUC={auc:.4f}", color="#4C78A8", linewidth=2)
    plt.plot([0, 1], [0, 1], "--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"{model_name} - ROC Curve")
    plt.legend()
    save_figure(f"{slugify_model_name(model_name)}_roc.png")


def plot_single_pr(model_name, y_true, y_prob):
    '''Save a per-model Precision-Recall curve.'''
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    pr_auc = average_precision_score(y_true, y_prob)
    plt.figure()
    plt.plot(recall, precision, label=f"PR-AUC={pr_auc:.4f}", color="#F58518", linewidth=2)
    plt.axhline(np.mean(y_true), linestyle="--", color="gray", label="Base rate")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"{model_name} - Precision-Recall Curve")
    plt.legend()
    save_figure(f"{slugify_model_name(model_name)}_pr.png")


def plot_single_ks(model_name, y_true, y_prob):
    '''Save a KS curve based on ROC thresholds.'''
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    finite = np.isfinite(thresholds)
    thresholds, fpr, tpr = thresholds[finite], fpr[finite], tpr[finite]
    order = np.argsort(thresholds)
    thresholds, fpr, tpr = thresholds[order], fpr[order], tpr[order]
    ks_values = tpr - fpr
    plt.figure()
    plt.plot(thresholds, tpr, label="TPR", color="#E45756")
    plt.plot(thresholds, fpr, label="FPR", color="#4C78A8")
    plt.plot(thresholds, ks_values, label=f"TPR-FPR (KS={np.max(ks_values):.4f})", color="#54A24B")
    plt.xlabel("Probability Threshold")
    plt.ylabel("Rate")
    plt.title(f"{model_name} - KS Curve")
    plt.legend()
    save_figure(f"{slugify_model_name(model_name)}_ks.png")


def plot_confusion(model_name, y_true, y_prob, threshold):
    '''Save the confusion matrix at the selected threshold.'''
    matrix = confusion_matrix(y_true, np.asarray(y_prob) >= threshold, labels=[0, 1])
    plt.figure(figsize=(6, 5))
    plt.imshow(matrix, cmap="Blues")
    plt.colorbar()
    plt.xticks([0, 1], ["Predicted 0", "Predicted 1"])
    plt.yticks([0, 1], ["Actual 0", "Actual 1"])
    for (row, column), value in np.ndenumerate(matrix):
        plt.text(column, row, f"{value:,}", ha="center", va="center")
    plt.xlabel("Prediction")
    plt.ylabel("Actual")
    plt.title(f"{model_name} - Confusion Matrix")
    save_figure(f"{slugify_model_name(model_name)}_confusion.png")


def plot_single_calibration(model_name, y_true, y_prob):
    '''Save a per-model probability calibration curve.'''
    observed, predicted = calibration_curve(y_true, y_prob, n_bins=10, strategy="quantile")
    plt.figure()
    plt.plot(predicted, observed, marker="o", label=model_name, color="#B279A2")
    plt.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
    plt.xlabel("Mean Predicted Probability")
    plt.ylabel("Observed Bad Rate")
    plt.title(f"{model_name} - Calibration Curve")
    plt.legend()
    save_figure(f"{slugify_model_name(model_name)}_calibration.png")


def plot_probability_distribution(model_name, y_true, y_prob):
    '''Save predicted probability distributions for good and bad applicants.'''
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    plt.figure()
    plt.hist(y_prob[y_true == 0], bins=40, alpha=0.65, density=True, label="TARGET=0", color="#4C78A8")
    plt.hist(y_prob[y_true == 1], bins=40, alpha=0.65, density=True, label="TARGET=1", color="#E45756")
    plt.xlabel("Predicted Probability")
    plt.ylabel("Density")
    plt.title(f"{model_name} - Probability Distribution")
    plt.legend()
    save_figure(f"{slugify_model_name(model_name)}_probability_distribution.png")


def extract_feature_importance(model_name, model) -> pd.DataFrame:
    '''Extract Logistic Regression coefficients or built-in tree importance.'''
    feature_names = logistic_feature_names if model_name == "Logistic Regression" else tree_feature_names
    if model_name == "Logistic Regression":
        values = np.asarray(model.coef_[0])
        frame = pd.DataFrame({
            "Model": model_name,
            "Feature": feature_names,
            "Importance": np.abs(values),
            "SignedValue": values,
            "Direction": np.where(values >= 0, "Higher risk", "Lower risk"),
        })
    else:
        values = np.asarray(model.feature_importances_)
        frame = pd.DataFrame({
            "Model": model_name,
            "Feature": feature_names,
            "Importance": values,
            "SignedValue": np.nan,
            "Direction": "Non-directional importance",
        })
    return frame.sort_values("Importance", ascending=False).reset_index(drop=True)


def plot_feature_importance(model_name, importance_frame, top_n=20):
    '''Save the top-N coefficients or feature importances.'''
    top = importance_frame.head(top_n).sort_values("Importance")
    plt.figure(figsize=(10, 7))
    plt.barh(top["Feature"], top["Importance"], color="#72B7B2")
    plt.xlabel("Absolute Coefficient" if model_name == "Logistic Regression" else "Feature Importance")
    plt.ylabel("Feature")
    plt.title(f"{model_name} - Top {top_n} Features")
    save_figure(f"{slugify_model_name(model_name)}_feature_importance.png")

In [ ]:
importance_frames = []
for model_name, model in models.items():
    test_probability = probabilities[model_name]["test"]
    selected_threshold = float(
        model_metrics.loc[model_metrics["Model"] == model_name, "Threshold"].iloc[0]
    )
    plot_single_roc(model_name, y_test, test_probability)
    plot_single_pr(model_name, y_test, test_probability)
    plot_single_ks(model_name, y_test, test_probability)
    plot_confusion(model_name, y_test, test_probability, selected_threshold)
    plot_single_calibration(model_name, y_test, test_probability)
    plot_probability_distribution(model_name, y_test, test_probability)

    importance_frame = extract_feature_importance(model_name, model)
    importance_frames.append(importance_frame.head(50))
    importance_frame.to_csv(
        OUTPUT_DIR / f"{slugify_model_name(model_name)}_feature_importance.csv",
        index=False,
    )
    plot_feature_importance(model_name, importance_frame)

feature_importance_summary = pd.concat(
    [frame.dropna(axis=1, how="all") for frame in importance_frames],
    ignore_index=True,
)
feature_importance_summary.to_csv(OUTPUT_DIR / "feature_importance_summary.csv", index=False)

tree_model = models["Decision Tree"]
plt.figure(figsize=(22, 10))
plot_tree(
    tree_model,
    feature_names=tree_feature_names,
    class_names=["Good", "Bad"],
    max_depth=3,
    filled=True,
    proportion=True,
    rounded=False,
    fontsize=6,
)
plt.title(f"Decision Tree - First 3 Levels (Full depth={tree_model.get_depth()})")
save_figure("decision_tree_first_3_levels.png")

display(
    feature_importance_summary.groupby("Model", group_keys=False)
    .head(10)
    .reset_index(drop=True)
)

## 14. Five-Model Comparison Charts

Each comparison chart is saved separately to avoid crowded panels. Probability metrics use untouched test probabilities, while threshold metrics use the maximum-F1 threshold selected on the separate threshold partition.


In [ ]:
model_colors = {
    "Logistic Regression": "#4C78A8",
    "Decision Tree": "#F58518",
    "Random Forest": "#54A24B",
    "XGBoost": "#E45756",
    "LightGBM": "#B279A2",
}

plt.figure()
for model_name in models:
    fpr, tpr, _ = roc_curve(y_test, probabilities[model_name]["test"])
    auc = roc_auc_score(y_test, probabilities[model_name]["test"])
    plt.plot(fpr, tpr, label=f"{model_name} ({auc:.4f})", color=model_colors[model_name])
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Five Models - ROC Curves")
plt.legend(fontsize=8)
save_figure("comparison_roc_curves.png", show=True)

plt.figure()
for model_name in models:
    precision, recall, _ = precision_recall_curve(y_test, probabilities[model_name]["test"])
    pr_auc = average_precision_score(y_test, probabilities[model_name]["test"])
    plt.plot(recall, precision, label=f"{model_name} ({pr_auc:.4f})", color=model_colors[model_name])
plt.axhline(y_test.mean(), linestyle="--", color="gray", label="Base rate")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Five Models - Precision-Recall Curves")
plt.legend(fontsize=8)
save_figure("comparison_pr_curves.png", show=True)


def plot_metric_comparison(frame, metric, ascending=False, color="#4C78A8"):
    '''Save a five-model bar chart for one metric.'''
    ordered = frame.sort_values(metric, ascending=ascending)
    plt.figure(figsize=(9, 5.5))
    plt.bar(ordered["Model"], ordered[metric], color=color)
    plt.ylabel(metric)
    plt.xlabel("Model")
    plt.title(f"Five Models - {metric} Comparison")
    plt.xticks(rotation=20, ha="right")
    for index, value in enumerate(ordered[metric]):
        plt.text(index, value, f"{value:.4f}", ha="center", va="bottom", fontsize=8)
    save_figure(f"comparison_{metric.lower().replace('-', '_')}.png", show=True)


for metric, color, ascending in [
    ("AUC", "#4C78A8", False),
    ("AR", "#72B7B2", False),
    ("KS", "#54A24B", False),
    ("F1", "#F58518", False),
    ("Brier", "#E45756", True),
    ("TrainTime", "#B279A2", True),
    ("PredictTime", "#FF9DA6", True),
]:
    plot_metric_comparison(model_metrics, metric, ascending=ascending, color=color)

## 15. Risk-Decile Analysis

Test predictions are divided into ten bands from lowest to highest risk. The analysis focuses on bad-rate monotonicity, high-risk lift, and cumulative bad-applicant capture starting from the highest-risk band.


In [ ]:
def build_decile_table(model_name, y_true, y_prob) -> pd.DataFrame:
    '''Build a risk-decile table from low to high risk.'''
    work = pd.DataFrame({"actual": np.asarray(y_true), "probability": np.asarray(y_prob)})
    work["RiskDecile"] = pd.qcut(
        work["probability"].rank(method="first"),
        10,
        labels=list(range(1, 11)),
    ).astype(int)
    summary = work.groupby("RiskDecile", observed=True).agg(
        SampleCount=("actual", "size"),
        BadCount=("actual", "sum"),
        ActualBadRate=("actual", "mean"),
        MeanPredictedPD=("probability", "mean"),
    ).reset_index()
    summary["SampleShare"] = summary["SampleCount"] / len(work)
    summary["Lift"] = summary["ActualBadRate"] / work["actual"].mean()
    high_to_low_bad = summary.sort_values("RiskDecile", ascending=False)["BadCount"].cumsum()
    cumulative_map = dict(zip(
        summary.sort_values("RiskDecile", ascending=False)["RiskDecile"],
        high_to_low_bad / summary["BadCount"].sum(),
    ))
    summary["CumulativeBadCaptureHighToLow"] = summary["RiskDecile"].map(cumulative_map)
    summary["RiskLevel"] = pd.cut(
        summary["RiskDecile"],
        bins=[0, 3, 7, 10],
        labels=["Low", "Medium", "High"],
    ).astype(str)
    summary.insert(0, "Model", model_name)
    return summary


def plot_decile_outputs(model_name, table, show=False):
    '''Save bad-rate, lift, cumulative-capture, and predicted-versus-actual PD charts.'''
    slug = slugify_model_name(model_name)

    plt.figure()
    plt.plot(table["RiskDecile"], table["ActualBadRate"], marker="o", color="#E45756")
    plt.xlabel("Risk Decile (1=Low, 10=High)")
    plt.ylabel("Actual Bad Rate")
    plt.title(f"{model_name} - Decile Bad Rate")
    save_figure(f"{slug}_decile_bad_rate.png", show=show)

    plt.figure()
    plt.bar(table["RiskDecile"], table["Lift"], color="#54A24B")
    plt.axhline(1.0, linestyle="--", color="gray")
    plt.xlabel("Risk Decile (1=Low, 10=High)")
    plt.ylabel("Lift")
    plt.title(f"{model_name} - Decile Lift")
    save_figure(f"{slug}_decile_lift.png", show=show)

    descending = table.sort_values("RiskDecile", ascending=False).copy()
    descending["CumulativePopulation"] = descending["SampleShare"].cumsum()
    descending["CumulativeBadCapture"] = descending["BadCount"].cumsum() / descending["BadCount"].sum()
    plt.figure()
    plt.plot(descending["CumulativePopulation"], descending["CumulativeBadCapture"], marker="o")
    plt.plot([0, 1], [0, 1], "--", color="gray")
    plt.xlabel("Cumulative Population from Highest Risk")
    plt.ylabel("Cumulative Bad Capture")
    plt.title(f"{model_name} - Cumulative Bad Capture")
    save_figure(f"{slug}_cumulative_bad_capture.png", show=show)

    plt.figure()
    plt.plot(table["RiskDecile"], table["MeanPredictedPD"], marker="o", label="Mean Predicted PD")
    plt.plot(table["RiskDecile"], table["ActualBadRate"], marker="s", label="Actual Bad Rate")
    plt.xlabel("Risk Decile (1=Low, 10=High)")
    plt.ylabel("Rate")
    plt.title(f"{model_name} - Predicted PD vs Actual Bad Rate")
    plt.legend()
    save_figure(f"{slug}_decile_calibration.png", show=show)

In [ ]:
decile_tables = []
decile_quality_rows = []
for model_name in models:
    table = build_decile_table(model_name, y_test, probabilities[model_name]["test"])
    decile_tables.append(table)
    adjacent_increase_rate = float((np.diff(table["ActualBadRate"]) >= 0).mean())
    top_decile_capture = float(
        table.loc[table["RiskDecile"] == 10, "BadCount"].iloc[0] / table["BadCount"].sum()
    )
    decile_quality_rows.append({
        "Model": model_name,
        "AdjacentMonotonicIncreaseRate": adjacent_increase_rate,
        "TopDecileBadCapture": top_decile_capture,
        "TopDecileLift": float(table.loc[table["RiskDecile"] == 10, "Lift"].iloc[0]),
    })
    plot_decile_outputs(model_name, table, show=False)

decile_analysis = pd.concat(decile_tables, ignore_index=True)
decile_quality = pd.DataFrame(decile_quality_rows).sort_values(
    ["AdjacentMonotonicIncreaseRate", "TopDecileBadCapture"], ascending=False
)
decile_analysis.to_csv(OUTPUT_DIR / "decile_analysis.csv", index=False)
decile_quality.to_csv(OUTPUT_DIR / "decile_quality_summary.csv", index=False)
display(decile_quality.round(4))

best_selection_model = performance_gaps.sort_values("ModelSelectionAUC", ascending=False)["Model"].iloc[0]
plot_decile_outputs(
    best_selection_model,
    decile_analysis[decile_analysis["Model"] == best_selection_model],
    show=True,
)

## 16. Probability Calibration and Class-Prior Context

Balanced class weights and `scale_pos_weight` change the effective event prior seen during fitting. Their raw `predict_proba` outputs are therefore ranking scores, not portfolio PD estimates. This section compares four mappings for the two leading models: Raw, analytical prior-probability correction, Platt scaling, and Isotonic regression.

The independent calibration partition is divided into fit and method-selection halves. After a mapping is selected by calibration-selection Brier score, it is refitted on the complete calibration partition. The operating threshold is then selected on the separate threshold partition, and the final test partition is evaluated once. Calibration-in-the-large, calibration slope, observed/expected ratio, Brier score, and Log Loss are reported. The public data cannot support through-the-cycle anchoring because no long-run portfolio default rate is supplied.


In [ ]:
top_two_models = (
    performance_gaps.sort_values("ModelSelectionAUC", ascending=False)
    .head(2)["Model"]
    .tolist()
)
calibration_positions = np.arange(len(y_calibration))
calibration_fit_positions, calibration_select_positions = train_test_split(
    calibration_positions,
    test_size=0.5,
    stratify=np.asarray(y_calibration),
    random_state=RANDOM_STATE,
)
portfolio_bad_rate = float(y_train.mean())


def fit_probability_mapping(method, fit_probability, fit_y):
    if method == "Raw":
        return lambda values: np.asarray(values, dtype=float)
    if method == "PriorCorrection":
        return lambda values: prior_probability_correction(values, portfolio_bad_rate, 0.5)
    if method == "Platt":
        fitted = LogisticRegression(C=1e6, solver="lbfgs", max_iter=2_000, random_state=RANDOM_STATE)
        fitted.fit(np.asarray(fit_probability).reshape(-1, 1), np.asarray(fit_y))
        return lambda values: fitted.predict_proba(np.asarray(values).reshape(-1, 1))[:, 1]
    if method == "Isotonic":
        fitted = IsotonicRegression(out_of_bounds="clip")
        fitted.fit(np.asarray(fit_probability), np.asarray(fit_y))
        return lambda values: fitted.predict(np.asarray(values))
    raise ValueError(method)


calibration_methods = ("Raw", "PriorCorrection", "Platt", "Isotonic")
calibration_rows = []
selection_rows = []
selected_calibrated_test_probabilities = {}
selected_calibrated_threshold_probabilities = {}
diagnostic_rows = []

for model_name in top_two_models:
    raw_calibration = probabilities[model_name]["calibration"]
    raw_threshold = probabilities[model_name]["threshold"]
    raw_test = probabilities[model_name]["test"]
    method_selection = []
    for method in calibration_methods:
        mapping = fit_probability_mapping(
            method,
            raw_calibration[calibration_fit_positions],
            np.asarray(y_calibration)[calibration_fit_positions],
        )
        selection_probability = np.clip(
            mapping(raw_calibration[calibration_select_positions]), 1e-7, 1 - 1e-7
        )
        method_selection.append({
            "Model": model_name,
            "Calibration": method,
            "CalibrationSelectionBrier": brier_score_loss(
                np.asarray(y_calibration)[calibration_select_positions], selection_probability
            ),
            "CalibrationSelectionLogLoss": log_loss(
                np.asarray(y_calibration)[calibration_select_positions], selection_probability
            ),
        })
    selection_frame = pd.DataFrame(method_selection).sort_values(
        ["CalibrationSelectionBrier", "CalibrationSelectionLogLoss"]
    )
    selection_rows.extend(method_selection)
    selected_method = str(selection_frame.iloc[0]["Calibration"])

    full_mappings = {
        method: fit_probability_mapping(method, raw_calibration, y_calibration)
        for method in calibration_methods
    }
    test_by_method = {}
    for method, mapping in full_mappings.items():
        threshold_probability = np.clip(mapping(raw_threshold), 1e-7, 1 - 1e-7)
        test_probability = np.clip(mapping(raw_test), 1e-7, 1 - 1e-7)
        test_by_method[method] = test_probability
        selected_threshold = select_thresholds(y_threshold, threshold_probability)["max_f1"]
        test_metrics = evaluate_model(model_name, y_test, test_probability, selected_threshold)
        selection_result = next(row for row in method_selection if row["Calibration"] == method)
        calibration_rows.append({
            "Model": model_name,
            "Calibration": method,
            "SelectedForFinalUse": method == selected_method,
            "CalibrationSelectionBrier": selection_result["CalibrationSelectionBrier"],
            "Threshold": selected_threshold,
            "AUC": test_metrics["AUC"],
            "AR": test_metrics["AR"],
            "KS": test_metrics["KS"],
            "PR-AUC": test_metrics["PR-AUC"],
            "F1": test_metrics["F1"],
            "Brier": test_metrics["Brier"],
            "LogLoss": test_metrics["LogLoss"],
        })
        diagnostic_rows.append(calibration_diagnostics(y_test, test_probability, model_name, method))

    selected_calibrated_test_probabilities[model_name] = test_by_method[selected_method]
    selected_calibrated_threshold_probabilities[model_name] = np.clip(
        full_mappings[selected_method](raw_threshold), 1e-7, 1 - 1e-7
    )

    plt.figure()
    for method in calibration_methods:
        observed, predicted = calibration_curve(
            y_test,
            test_by_method[method],
            n_bins=10,
            strategy="quantile",
        )
        suffix = " (selected)" if method == selected_method else ""
        plt.plot(predicted, observed, marker="o", label=f"{method}{suffix}")
    plt.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")
    plt.xlabel("Mean Predicted Probability")
    plt.ylabel("Observed Bad Rate")
    plt.title(f"{model_name} - Calibration Methods")
    plt.legend()
    save_figure(f"{slugify_model_name(model_name)}_calibration_methods.png", show=True)

calibration_results = pd.DataFrame(calibration_rows)
calibration_selection = pd.DataFrame(selection_rows)
selected_calibration = calibration_results.loc[
    calibration_results["SelectedForFinalUse"],
    ["Model", "Calibration", "CalibrationSelectionBrier", "Brier", "LogLoss"],
].rename(columns={"Calibration": "SelectedCalibration"})
calibration_diagnostic_table = pd.DataFrame(diagnostic_rows)
cost_sensitivity = cost_sensitivity_table(y_test, selected_calibrated_test_probabilities)

calibration_results.to_csv(OUTPUT_DIR / "calibration_results.csv", index=False)
calibration_selection.to_csv(OUTPUT_DIR / "calibration_selection_results.csv", index=False)
selected_calibration.to_csv(OUTPUT_DIR / "selected_calibration_methods.csv", index=False)
calibration_diagnostic_table.to_csv(OUTPUT_DIR / "calibration_diagnostics.csv", index=False)
cost_sensitivity.to_csv(OUTPUT_DIR / "cost_sensitivity.csv", index=False)
display(calibration_results.round(4))
display(calibration_diagnostic_table.round(4))
display(cost_sensitivity.round(4))


## 17. Statistical Uncertainty and Paired Model Comparison

Stratified bootstrap intervals quantify final-test uncertainty for AUC, KS, and PR-AUC. Paired bootstrap differences retain the same resampled applicants for both models. A paired two-sided DeLong test is reported for the two models selected by model-selection AUC. These procedures evaluate uncertainty conditional on the fitted models; they do not replace repeated model-development splits or out-of-time testing.


In [ ]:
final_test_probabilities = {
    model_name: values["test"] for model_name, values in probabilities.items()
}
bootstrap_intervals, paired_bootstrap_differences = bootstrap_metric_intervals(
    y_test,
    final_test_probabilities,
    n_bootstrap=BOOTSTRAP_REPLICATES,
    random_state=RANDOM_STATE,
)
leading_models = (
    performance_gaps.sort_values("ModelSelectionAUC", ascending=False)
    .head(2)["Model"]
    .tolist()
)
delong_results = delong_auc_test(
    y_test,
    final_test_probabilities[leading_models[0]],
    final_test_probabilities[leading_models[1]],
    leading_models[0],
    leading_models[1],
)
bootstrap_intervals.to_csv(OUTPUT_DIR / "bootstrap_metric_intervals.csv", index=False)
paired_bootstrap_differences.to_csv(OUTPUT_DIR / "paired_bootstrap_auc_differences.csv", index=False)
delong_results.to_csv(OUTPUT_DIR / "delong_auc_test.csv", index=False)

auc_intervals = bootstrap_intervals.loc[bootstrap_intervals["Metric"] == "AUC"].sort_values("Estimate")
plt.figure(figsize=(9, 5.5))
positions = np.arange(len(auc_intervals))
lower_error = auc_intervals["Estimate"] - auc_intervals["Lower"]
upper_error = auc_intervals["Upper"] - auc_intervals["Estimate"]
plt.errorbar(
    auc_intervals["Estimate"], positions,
    xerr=np.vstack([lower_error, upper_error]),
    fmt="o", capsize=4, color="#4C78A8",
)
plt.yticks(positions, auc_intervals["Model"])
plt.xlabel("ROC-AUC with bootstrap 95% interval")
plt.title("Final-Test AUC Uncertainty")
save_figure("bootstrap_auc_intervals.png", show=True)
display(bootstrap_intervals.round(4))
display(paired_bootstrap_differences.round(4))
display(delong_results.round(4))


## 18. Customer-Segment and Random-Split Stability Diagnostics

Segment AUC is reported only where each segment has at least 25 events and 25 non-events. PSI compares the random training and final-test partitions using training-derived bins. Because both samples are random draws from the same table, low PSI is only a pipeline consistency check; it is not evidence of temporal stability.


In [ ]:
age_years_test = -X_test["DAYS_BIRTH"] / 365.25
income_edges = np.unique(np.quantile(X_train["AMT_INCOME_TOTAL"].dropna(), [0, 0.25, 0.5, 0.75, 1]))
income_edges[0], income_edges[-1] = -np.inf, np.inf
segment_frame = pd.DataFrame({
    "Gender": X_test["CODE_GENDER"].astype("string"),
    "ContractType": X_test["NAME_CONTRACT_TYPE"].astype("string"),
    "AgeBand": pd.cut(age_years_test, bins=[20, 30, 40, 50, 60, 70, 100], right=False).astype("string"),
    "IncomeQuartile": pd.cut(
        X_test["AMT_INCOME_TOTAL"], bins=income_edges,
        labels=["Q1", "Q2", "Q3", "Q4"], include_lowest=True,
    ).astype("string"),
}).reset_index(drop=True)
segment_results = segment_performance_table(
    y_test,
    final_test_probabilities,
    segment_frame,
)
segment_results.to_csv(OUTPUT_DIR / "segment_performance.csv", index=False)

numeric_psi_columns = [
    column for column in selected_columns
    if column in X_train.columns and pd.api.types.is_numeric_dtype(X_train[column])
]
feature_psi = build_psi_table(X_train, X_test, numeric_psi_columns)
score_psi = pd.DataFrame([
    {
        "Feature": f"SCORE::{model_name}",
        "PSI": population_stability_index(probabilities[model_name]["train"], probability),
        "Comparison": "random_test_vs_train",
        "Interpretation": "random-split score consistency only",
    }
    for model_name, probability in final_test_probabilities.items()
])
psi_results = pd.concat([feature_psi, score_psi], ignore_index=True).sort_values("PSI", ascending=False)
psi_results.to_csv(OUTPUT_DIR / "random_split_psi.csv", index=False)

top_psi = psi_results.head(15).sort_values("PSI")
plt.figure(figsize=(9, 6))
plt.barh(top_psi["Feature"], top_psi["PSI"], color="#72B7B2")
plt.xlabel("PSI: random test versus training")
plt.title("Largest Random-Split PSI Diagnostics")
save_figure("random_split_psi.png", show=True)

leading_segment_results = segment_results.loc[
    segment_results["Model"].isin(leading_models) & segment_results["AUCReported"]
].copy()
leading_segment_results["Segment"] = (
    leading_segment_results["SegmentVariable"] + ": " + leading_segment_results["SegmentValue"]
)
pivot = leading_segment_results.pivot(index="Segment", columns="Model", values="AUC").dropna()
pivot.sort_index().plot(kind="barh", figsize=(10, max(6, 0.35 * len(pivot))))
plt.xlabel("Segment ROC-AUC")
plt.title("Leading Models by Customer Segment")
plt.legend(fontsize=8)
save_figure("segment_auc_leading_models.png", show=True)
display(segment_results.round(4))
display(psi_results.head(20).round(4))


## 19. Reproducible English Analysis Report

The Markdown report below reads its values from the current run. Its conclusions are deliberately conditional: the final test is one random holdout, and the dataset cannot support out-of-time stability, reject inference, long-run PD anchoring, LGD/EAD loss estimation, or regulatory capital conclusions.


In [ ]:
def dataframe_to_markdown(frame: pd.DataFrame, float_digits=4) -> str:
    formatted = frame.copy()
    for column in formatted.columns:
        if pd.api.types.is_float_dtype(formatted[column]):
            formatted[column] = formatted[column].map(
                lambda value: "" if pd.isna(value) else f"{value:.{float_digits}f}"
            )
    headers = [str(column) for column in formatted.columns]
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    for row in formatted.astype(str).itertuples(index=False, name=None):
        lines.append("| " + " | ".join(row) + " |")
    return "\n".join(lines)


result_report_table = model_metrics[[
    "Model", "AUC", "KS", "PR-AUC", "F1", "Brier", "LogLoss", "TrainTime", "TuneTime"
]].copy()
auc_interval_table = bootstrap_intervals.loc[
    bootstrap_intervals["Metric"] == "AUC", ["Model", "Estimate", "Lower", "Upper"]
].sort_values("Estimate", ascending=False)
top_model = str(model_metrics.iloc[0]["Model"])
runner_up = str(model_metrics.iloc[1]["Model"])
top_delong = delong_results.iloc[0]
delong_model_a = str(top_delong["ModelA"])
delong_model_b = str(top_delong["ModelB"])
statistically_distinguishable = bool(top_delong["PValueTwoSided"] < 0.05)
distinction_text = (
    "is statistically distinguishable at the 5% level under the paired DeLong test"
    if statistically_distinguishable
    else "is not statistically distinguishable at the 5% level under the paired DeLong test"
)
top_interval = auc_interval_table.loc[auc_interval_table["Model"] == top_model].iloc[0]
runner_interval = auc_interval_table.loc[auc_interval_table["Model"] == runner_up].iloc[0]
maximum_random_split_psi = float(psi_results["PSI"].max())

model_tradeoffs = pd.DataFrame([
    {"Model": "Logistic Regression", "PrimaryStrength": "Transparent directional benchmark", "PrimaryLimitation": "Linear specification unless transformations are added"},
    {"Model": "Decision Tree", "PrimaryStrength": "Readable nonlinear rules", "PrimaryLimitation": "Coarse probabilities and unstable partitions"},
    {"Model": "Random Forest", "PrimaryStrength": "Bagged nonlinear interactions", "PrimaryLimitation": "Large train-selection gap and higher explanation cost"},
    {"Model": "XGBoost", "PrimaryStrength": "Strong tabular discrimination", "PrimaryLimitation": "Longer resource-constrained screening and fit time"},
    {"Model": "LightGBM", "PrimaryStrength": "Efficient boosted-tree challenger", "PrimaryLimitation": "Leaf-wise growth requires explicit complexity controls"},
])
model_tradeoffs.to_csv(OUTPUT_DIR / "model_tradeoffs.csv", index=False)

report = f'''# Home Credit Default Risk: Model Development Evidence

## Executive Summary

Five models were developed on mutually exclusive training, model-selection, calibration, threshold, and final-test partitions. On the final random holdout, {top_model} has the highest observed AUC ({top_interval['Estimate']:.4f}; bootstrap 95% CI {top_interval['Lower']:.4f}-{top_interval['Upper']:.4f}), followed by {runner_up} ({runner_interval['Estimate']:.4f}; {runner_interval['Lower']:.4f}-{runner_interval['Upper']:.4f}). The pre-specified DeLong comparison uses the two candidates selected by model-selection AUC: {delong_model_a} versus {delong_model_b}, with an AUC difference of {top_delong['AUCDifference']:.4f} (p={top_delong['PValueTwoSided']:.4f}); this difference {distinction_text}. The evidence supports a short list of leading candidates, not a claim of universal algorithm superiority.

## Data and Experimental Design

- Public Home Credit application records: {len(df_raw):,}; observed event rate: {y.mean():.2%}.
- Training/model-selection/calibration/threshold/test shares: 60%/10%/10%/10%/10%.
- Feature filtering and preprocessing are fitted on training data only.
- Parameter screening is deliberately limited to the candidate counts reported below.
- Boosting candidates use early stopping on model-selection data; selected iteration counts are then refitted on the complete training partition.
- Final training time is measured from this clean complete-training fit for every model.

## Final Test Results

{dataframe_to_markdown(result_report_table)}

## Statistical Uncertainty

{dataframe_to_markdown(auc_interval_table)}

Paired bootstrap differences and the DeLong test are saved separately. Bootstrap intervals quantify uncertainty conditional on this fitted sample and model. They do not measure performance variability across new booking periods or fully repeated model-development splits.

## Probability Calibration in Credit-Risk Context

Balanced class weights and `scale_pos_weight` alter the effective training prior. Raw boosted-model probabilities therefore overstate portfolio PD levels and are treated as ranking scores. The notebook compares analytical prior correction, Platt scaling, and Isotonic regression on an independent calibration sample, selects the mapping without test data, and reports calibration-in-the-large, calibration slope, observed/expected ratio, Brier score, and Log Loss.

The observed bad rate in this public sample is a point-in-time sample rate, not a through-the-cycle long-run default rate. No long-run portfolio anchor is available, so the project does not claim TTC calibration, grade-level long-run calibration, or regulatory PD estimation.

## Segment and Drift Diagnostics

Segment AUC and bad rates are reported for gender, contract type, age band, and income band subject to minimum event counts. PSI compares the random training and test partitions only. The maximum observed random-split PSI is {maximum_random_split_psi:.4f}; this is a pipeline consistency diagnostic and is not evidence of temporal stability.

## Decision Costs

The threshold cost table uses explicit illustrative ratios between approving a bad applicant and rejecting a good applicant. It is a sensitivity analysis, not an expected-loss, profitability, capital, or pricing model. Real strategy selection requires lender-specific LGD, EAD, margin, funding cost, operating cost, capacity, and capital assumptions.

## Scope Boundaries

- No reliable application timestamp is supplied, so out-of-time validation is not performed.
- Labels are unavailable for Kaggle's official test file; the final test is a reserved random holdout.
- No accepted/rejected application outcome mechanism is supplied, so reject inference is not performed.
- Bureau, previous-application, and repayment-history aggregation is not implemented in this repository and is not represented as a completed module.
- Built-in coefficients and feature importance are descriptive. SHAP analysis is not claimed or silently substituted.
- A single development split cannot establish ranking stability across random seeds or future customer vintages.

## Model-Selection Position

{top_model} is the observed leader on this final holdout, while {runner_up} remains close in observed AUC. A production champion should be chosen only after out-of-time testing, repeated development samples, segment review, calibration anchoring, reason-code review, and economic strategy analysis. Logistic Regression remains the transparent benchmark.
'''

report_path = OUTPUT_DIR / "model_comparison_report.md"
report_path.write_text(textwrap.dedent(report).strip() + "\n", encoding="utf-8")
display(Markdown(report))
print(f"English report saved: {report_path}")


## 20. Reproducibility and Model-Risk Checks

These checks focus on evidence integrity rather than output volume: disjoint split identities, metric reproduction from saved predictions, model reload agreement, active LightGBM bagging, warning review, required statistical outputs, and the absence of target or identifier leakage.


In [ ]:
warning_log = pd.DataFrame(warning_records, columns=["Context", "Category", "Message"])
warning_log.to_csv(OUTPUT_DIR / "warnings_log.csv", index=False)
critical_warning_mask = (
    warning_log["Category"].str.contains("ConvergenceWarning", case=False, na=False)
    | warning_log["Message"].str.contains("ignored|not used|has no effect", case=False, na=False)
) if len(warning_log) else pd.Series([], dtype=bool)

expected_model_files = {
    MODEL_DIR / f"{slugify_model_name(model_name)}.joblib" for model_name in models
} | {MODEL_DIR / "logistic_preprocessor.joblib", MODEL_DIR / "tree_preprocessor.joblib"}
reload_max_differences = {}
for model_name in models:
    reloaded = joblib.load(MODEL_DIR / f"{slugify_model_name(model_name)}.joblib")
    reloaded_probability = positive_probability(reloaded, matrix_sets[model_name][-1])
    reload_max_differences[model_name] = float(
        np.max(np.abs(reloaded_probability - probabilities[model_name]["test"]))
    )

reproduced_auc = {
    model_name: roc_auc_score(test_predictions[TARGET_COL], test_predictions[model_name])
    for model_name in models
}
recorded_auc = model_metrics.set_index("Model")["AUC"].to_dict()

required_output_files = [
    OUTPUT_DIR / "model_metrics.csv",
    OUTPUT_DIR / "test_predictions.csv",
    OUTPUT_DIR / "bootstrap_metric_intervals.csv",
    OUTPUT_DIR / "paired_bootstrap_auc_differences.csv",
    OUTPUT_DIR / "delong_auc_test.csv",
    OUTPUT_DIR / "segment_performance.csv",
    OUTPUT_DIR / "random_split_psi.csv",
    OUTPUT_DIR / "calibration_diagnostics.csv",
    OUTPUT_DIR / "cost_sensitivity.csv",
    OUTPUT_DIR / "model_comparison_report.md",
]
lightgbm_parameters = json.loads(
    best_parameters.loc[best_parameters["Model"] == "LightGBM", "BestParameters"].iloc[0]
)
validation_checks = {
    "five_models_trained": set(models) == {"Logistic Regression", "Decision Tree", "Random Forest", "XGBoost", "LightGBM"},
    "development_splits_are_disjoint": all(
        id_sets[left].isdisjoint(id_sets[right])
        for left_index, left in enumerate(id_sets)
        for right in list(id_sets)[left_index + 1:]
    ),
    "all_core_metrics_finite": np.isfinite(
        model_metrics[["AUC", "AR", "KS", "PR-AUC", "F1", "Brier", "LogLoss"]].to_numpy()
    ).all(),
    "saved_predictions_reproduce_auc": all(
        np.isclose(reproduced_auc[name], recorded_auc[name], atol=1e-12) for name in models
    ),
    "reloaded_models_reproduce_predictions": max(reload_max_differences.values()) < 1e-12,
    "no_target_or_id_in_features": TARGET_COL not in selected_columns and ID_COL not in selected_columns,
    "lightgbm_subsample_frequency_active": int(lightgbm_parameters.get("subsample_freq", 0)) > 0,
    "no_critical_fit_warnings": not bool(critical_warning_mask.any()),
    "required_evidence_files_exist": all(path.exists() for path in required_output_files),
    "exact_expected_model_artifacts": set(MODEL_DIR.glob("*.joblib")) == expected_model_files,
    "bootstrap_intervals_cover_all_models": bootstrap_intervals["Model"].nunique() == len(models),
    "segment_results_cover_all_models": segment_results["Model"].nunique() == len(models),
}

validation_frame = pd.DataFrame({
    "check": validation_checks.keys(),
    "passed": validation_checks.values(),
})
validation_frame.to_csv(OUTPUT_DIR / "validation_checks.csv", index=False)
display(validation_frame)
display(warning_log)

failed_checks = validation_frame.loc[~validation_frame["passed"], "check"].tolist()
if failed_checks:
    raise AssertionError(f"Model-risk checks failed: {failed_checks}")

print("VALIDATION_CHECKS_PASSED=True")
print(f"Maximum model reload probability difference: {max(reload_max_differences.values()):.3e}")
print(f"Audited output directory: {OUTPUT_DIR}")
